# EduPro — 03. Personalized Recommendations & Evaluation

This notebook demonstrates the cluster-aware recommendation engine for a selected learner. It excludes courses already taken and scores candidates using preference fit, course rating, peer-segment popularity and overall popularity.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "dashboard").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from dashboard.data_loader import build_data
from Src.recommender import recommend_courses

D = build_data()
learners = D["learners"]
print("Available learners:", learners.UserID.nunique())
print("Available courses:", D["courses"].CourseID.nunique())

In [ ]:
# Choose a real learner ID from the workbook
display(learners[["UserID","segment_name","preferred_category","preferred_level",
                  "total_enrollments","category_diversity","learning_depth_index"]].head(10))
selected_user = str(learners.iloc[0]["UserID"])
print("Selected learner:", selected_user)

In [ ]:
# Generate personalized recommendations
recommendations = recommend_courses(
    uid=selected_user,
    learners=D["learners"],
    enriched=D["enriched"],
    popularity=D["popularity"],
    course_user=D["course_user"],
    category=None,
    level=None,
    limit=10
)
display(recommendations[["course_name","category","level","course_type",
                         "avg_rating","enrollments","match_score"]])

In [ ]:
# Verify that recommendations do not include courses already enrolled by the learner
already_taken = set(D["course_user"].loc[
    D["course_user"].UserID.astype(str)==selected_user, "CourseID"
].astype(str))
recommended_ids = set(recommendations["CourseID"].astype(str)) if not recommendations.empty else set()
assert already_taken.isdisjoint(recommended_ids), "A previously enrolled course was recommended."
print("No previously enrolled courses appear in the recommendation list.")

In [ ]:
# Explain the transparent heuristic score components
if not recommendations.empty:
    component_cols = ["category_match","level_match","rating_norm","peer_popularity","pop_norm","match_score"]
    display(recommendations[["course_name"] + component_cols].head(10))
else:
    print("No eligible unseen courses for this learner.")

In [ ]:
# Segment-level popularity evidence for the selected learner
profile = learners[learners.UserID.astype(str)==selected_user].iloc[0]
peer_ids = set(learners.loc[learners.segment_id==profile.segment_id, "UserID"].astype(str))
peer_enrollments = D["enriched"][D["enriched"].UserID.astype(str).isin(peer_ids)]
peer_category = peer_enrollments["CourseCategory"].value_counts(normalize=True).mul(100).round(2)
print("Selected segment:", profile.segment_name)
display(peer_category.rename_axis("Category").to_frame("Peer enrollment share (%)").head(10))

## Evaluation protocol and reporting boundaries

- **Recommendation precision:** historical transactions do not directly label whether a suggested course is relevant. Create a labelled relevance set or a temporal holdout protocol before reporting Precision@K/Recall@K.
- **Engagement lift:** requires observing engagement after recommendations, ideally through an A/B test. Do not claim causal lift from historical data.
- **Intra-cluster similarity:** report within-cluster feature distances or cohesion statistics alongside silhouette.
- The displayed recommendation score is a transparent heuristic relevance estimate, not a calibrated probability.